In [3]:
import pandas as pd 
import numpy as np 
import geopandas as gpd 
from pyproj import Transformer
import os
%matplotlib inline
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show
from os.path import isfile, join
from os import listdir
import time 
#import rioxarray as rxr
# from tqdm import tqdm
# from IPython.display import display, Javascript

# from pythontoolbox.transverse import args_util
# from pythontoolbox.transverse.dbs import mysql_util

import os

In [7]:
df = pd.read_csv('../analyse_clinique/data/data_cleaned/patients_FR_IDF_geocoded_adulte_clinique.csv',sep=";")
gdf = (gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.x, df.y)).set_crs(epsg=4326)).to_crs(epsg=27572)
gdf = gdf[['pseudo_provisoire','geometry']]

In [ ]:
gdf.to_feather(f'output/polluant_extraction_{0}.feather')

## Optimisation du code avec stockage du raster en mémoire : test pour 3 fichiers
- 1.05 sec pour un polluantpour QGIS 


In [8]:
test1 = {'pseudo_provisoire': [0,1,2,3,4],'date': ['2017-01-04','2017-01-04','2017-01-04','2017-01-04','2017-01-04'],'PM10':[18.375,19.75,15.9375,16.25,18.9375]}
         
test2 = {'pseudo_provisoire': [0,1,2,3,4],'date': ['2017-01-04','2017-01-04','2017-01-04','2017-01-04','2017-01-04'],'PM25':[17.375,29.75,16.9375,17.25,19.9375]}

test3 = {'pseudo_provisoire': [0,1,2,3,4],'date': ['2018-01-04','2018-01-04','2018-01-04','2018-01-04','2018-01-04'],'PM25':[17.375,29.75,16.9375,17.25,19.9375]}
test4 = {'pseudo_provisoire': [0,1,2,3,4],'date': ['2018-01-04','2018-01-04','2018-01-04','2018-01-04','2018-01-04'],'03':[27.375,49.75,17.9375,17.25,19.9375]}
test5 = {'pseudo_provisoire': [0,1,2,3,4],'date': ['2017-01-04','2017-01-04','2017-01-04','2017-01-04','2017-01-04'],'NO2':[27.375,29.75,16.9375,17.25,19.9375]}

         
df_conc1 = pd.concat([pd.DataFrame(test1),pd.DataFrame(test2)],axis=1)
df_conc2 = pd.concat([pd.DataFrame(test1),pd.DataFrame(test3)],axis=0)


df_conc1 = df_conc1.T.drop_duplicates().T
df_conc2 = df_conc2.T.drop_duplicates().T

In [10]:
df_test4 = pd.DataFrame(test5)
if df_test4['date'].unique() in df_conc2['date'].unique():
    pollDate = df_test4['date'].unique()[0]
    df_conc2.loc[df_conc2['date']==pollDate  ,'NO2'] = df_test4['NO2']
    
df_conc2

df_test1 = pd.DataFrame(test1)




,pseudo_provisoire,date,PM10,PM25,NO2
0,0,2017-01-04,18.375,NaN,27.3750
1,1,2017-01-04,19.75,NaN,29.7500
2,2,2017-01-04,15.9375,NaN,16.9375
3,3,2017-01-04,16.25,NaN,17.2500
4,4,2017-01-04,18.9375,NaN,19.9375
0,0,2018-01-04,NaN,17.375,NaN
1,1,2018-01-04,NaN,29.75,NaN
2,2,2018-01-04,NaN,16.9375,NaN
3,3,2018-01-04,NaN,17.25,NaN
4,4,2018-01-04,NaN,19.9375,NaN


In [11]:
df_conc2 = df_conc2[['pseudo_provisoire','date','PM10']] 
columns = df_conc2.columns.tolist()
columns.remove('pseudo_provisoire')
columns.remove('date')
col_poll = columns[0]

In [12]:
pollDate = df_test4['date'].unique()[0]
print(pollDate)
df_conc2.loc[df_conc2['date']==pollDate ,:] #= df_test4['03']

2017-01-04


,pseudo_provisoire,date,PM10
0,0,2017-01-04,18.375
1,1,2017-01-04,19.75
2,2,2017-01-04,15.9375
3,3,2017-01-04,16.25
4,4,2017-01-04,18.9375


In [13]:
df_test = df[['pseudo_provisoire','adresse','codepost']]


df_add = {'pseudo_provisoire': df_test['pseudo_provisoire'], 'adresse':df_test['adresse'], 'date':'2017-01-04'}
# df_test.loc[len(df)] = new_rows
df_add = pd.DataFrame(df_add)
# df_test
df_dropduplicate = pd.concat([df_test,df_add],axis=1)


df_noduplicate = df_dropduplicate.T.drop_duplicates().T

df_noduplicate

,pseudo_provisoire,adresse,codepost,date
0,1,34 RUE DES FRERES CHAUSSONS,92600.0,2017-01-04
1,2,11 RUE EMILE DUBOIS,75014.0,2017-01-04
2,3,48 CHEMIN VERT,78680.0,2017-01-04
3,5,31 RUE DU GENERAL DE MIRIBEL,92500.0,2017-01-04
4,6,124 RUE JEAN BAPTISTE CHARCOT,92400.0,2017-01-04
...,...,...,...,...
44887,64286,15 RUE HOUSSA OUAID,77500.0,2017-01-04
44888,64290,3 AV DE FOUILLEUSE,92210.0,2017-01-04
44889,64292,60 RUE BAUDRICOURT,75013.0,2017-01-04
44890,64293,159 AVENUE DE LA REPUBLIQUE,92320.0,2017-01-04


In [14]:
pollDate = '2018-01-04'

if (pollDate in df_noduplicate['date'].unique()) is False: 
    print('yes')

yes


#### Boucle sur nos patients

In [17]:
# Préparation des chemins et chargement initial
start_time = time.time()
dir_path = "../analyse_clinique/data/airparif_dir/airparif/"
files = [f for f in listdir(dir_path) if f.endswith('.nc')] 
gdf_patients = gdf.copy()

raster_time = time.time()
# Chargement de tous les rasters en mémoire
rasters = {}
for file in files:
    try:
        with rasterio.open(join(dir_path, file)) as src:
            rasters[file] = {
                'data': src.read(1),  # Charger les données raster
                'transform': src.transform  # Sauvegarder la transformation pour la localisation des points
            }
    except Exception as e:
        files_not_found.append(file)
    # if file == files[-1]:
raster_final = time.time()   
raster_final_time = raster_final - raster_time
print(f"temps de chargement raster : {raster_final_time} sec")

# Fonction pour traiter chaque point avec les rasters chargés
def process_point(point, rasters):
    results = {}
    for filename, raster in rasters.items():
        rasterName = filename
        pollName = rasterName.split('_')[0].split('/')[-1]
        pollDate = rasterName.split('_')[-1].split('.')[0]
        col_name = f"{pollName}_{pollDate}"

        # Calcul des indices de la grille pour le point
        row, col = rasterio.transform.rowcol(raster['transform'], point.x, point.y)
        # Extraction de la valeur du raster
        value = raster['data'][row, col]
        results[col_name] = value
    return results

gdf_time = time.time()
# Appliquer la fonction à chaque point du GeoDataFrame
for i, row in gdf_patients.iterrows():
    point = row['geometry']
    results = process_point(point, rasters)
    for col_name, value in results.items():
        gdf_patients.at[i, col_name] = value

gdf_final = time.time()        
gdf_final_time = gdf_final - gdf_time
print(f"temps d'extraction : {gdf_final_time} sec")

# gdf.to_file("../../data/airparif/patients_output_test.shp", driver='ESRI Shapefile')
gdf.to_file("../analyse_clinique/data/airparif_dir/airparif_test/output/patients_output_test.shp", driver='ESRI Shapefile')

end_time = time.time()
final_time = end_time - start_time
print(f"temps écoulé : {final_time} sec")#, {final_time/60} min")


temps de chargement raster : 0.000995635986328125 sec
temps d'extraction : 2.4496712684631348 sec


C:\Users\lpokambo\AppData\Local\Temp\ipykernel_13120\1205543569.py:53: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file("../analyse_clinique/data/airparif_dir/airparif_test/output/patients_output_test.shp", driver='ESRI Shapefile')


temps écoulé : 6.590215682983398 sec


In [ ]:
# fig, ax = plt.subplots(figsize=(12,12))
# pointData.plot(ax=ax, color='orangered')
# show(ndviRaster, ax=ax)   

In [18]:
pollDate = '20240502'

date = pollDate[:4] +'-' + pollDate[4:6] +'-' + pollDate[6:]
date

'2024-05-02'

## Application du code pour l'ensemble de nos fichiers : 

#### Boucle sur l'ensemble nos fichiers de pollution 

In [19]:

def process_raster_multibands(file_path, gdf):
    gdf_processed = gdf.copy() 
    # Open the NetCDF file with rioxarray
    ds = rxr.open_rasterio(file_path, masked=True)

    # List of variables to process
    variables = ['PM10', 'NO2', 'PM25']

    for var_name in variables:
        file_name = os.path.basename(file_path)
        output_raster_file = f"./temp_rasters/{file_name}_{var_name}.tif"
        # Save each variable as a raster
        ds[var_name].rio.to_raster(output_raster_file)

        # Open the saved raster file with rasterio
        with rasterio.open(output_raster_file) as raster:
            raster_data = raster.read(1)
            transform = raster.transform

            def get_raster_value(point):
                row, col = rasterio.transform.rowcol(transform, point.x, point.y)
                return raster_data[row, col]

            # Construct column name based on variable and date
            raster_name = os.path.basename(file_path)
            pollDate = raster_name.split('_')[-1].split('.')[0]
            
            gdf_processed['date'] = pollDate[:4] +'-' + pollDate[4:6] +'-' + pollDate[6:]
            
            # Apply function to each point in the GeoDataFrame
            gdf_processed[f'{var_name}_chron'] = gdf_processed['geometry'].apply(get_raster_value)
    return gdf_processed
    

def process_raster(file_path, gdf):
    gdf_processed = gdf.copy() 

    with rasterio.open(file_path) as raster:
        raster_data = raster.read(1)  # Charger les données raster
        transform = raster.transform  # Sauvegarder la transformation pour la localisation des points
        
        def get_raster_value(point):
            row, col = rasterio.transform.rowcol(transform, point.x, point.y)
            return raster_data[row, col]
        
        # Extraire le nom du polluant et la date du nom du fichier
        rasterName = os.path.basename(file_path)
        pollName = rasterName.split('_')[0]
        pollDate = rasterName.split('_')[-1].split('.')[0]

        gdf_processed['date'] = pollDate[:4] +'-' + pollDate[4:6] +'-' + pollDate[6:]
        
        # Appliquer la fonction à chaque point du GeoDataFrame
        gdf_processed[f'{pollName}'] = gdf_processed['geometry'].apply(get_raster_value)
        
    return gdf_processed


In [20]:
# mysql_config = args_util.Argument("test", "my_creds_mysql_path_vault")
# database_creds = mysql_config.get_creds()
# table_name = "table"
# dataframe = pd.DataFrame({"col1": [1, 2, 3], "col2": [3, 4, 5]})
# insert_bulk_dataframe(database_creds, table_name, dataframe)
#     >>> insert_dataframe(database_creds, table_name, dataframe)

dict_test = {'id_0':'1','id_1':'2'}
print(dict_test)
dict_test = dict_test.clear()
print(dict_test)

{'id_0': '1', 'id_1': '2'}
None


In [21]:
def insert_table(nb_file_done,dic_poll_par_date):
    dataframe = pd.concat(dic_poll_par_date,axis=0)[['pseudo_provisoire','date', 'PM25', 'PM10', 'NO2', 'O3']]
    table_name = '202405_patients_pollution'
    # if nb_file_done == 100 : 
    insert_bulk_dataframe(database_creds,table_name, dataframe)
    dic_poll_par_date.clear()
    
    # feather.write_feather(output_file, join(root_dir,f'output/patients_polluants_{nb_file_done}.feather'))
                                    

In [22]:
root_dir = '../analyse_clinique/data/airparif_dir/airparif/'     

dic_poll_par_date = {}
treated_paths = set()
treated_polls = set()
nb_file_done = 0 
for polldir in os.listdir(root_dir):
    poll_path = join(root_dir,polldir)
    
    if 'pollution' in polldir:
    
        for yeardir in os.listdir(poll_path):
            year_path = join(poll_path,yeardir)

            if os.path.isdir(year_path) : 
                
                if 'chronique' in yeardir : 

                    for poll in os.listdir(year_path) :               
                        path_to_poll = join(year_path,poll)
                        
                        if path_to_poll not in treated_paths:
                            print(f'Starting directory {path_to_poll}')
                            treated_paths.add(path_to_poll)

                            for file in tqdm(os.listdir(path_to_poll)):
                            
                                if file.endswith('.nc') and file not in treated_polls:  

                                    path_to_file = os.path.join(path_to_poll, file)

                                    if file.split('_')[0] =='horair':
                                        gdf_processed = process_raster_multibands(path_to_file, gdf)    
                                        dic_poll_par_date[file] = gdf_processed
                                        nb_file_done +=1 
                                    else :
                                        gdf_processed = process_raster(path_to_file, gdf)   
                                        dic_poll_par_date[file] = gdf_processed
                                        nb_file_done +=1 
                                        
                                    treated_polls.add(path_to_file)

                                if nb_file_done%100 == 0 : 
                                    # output_file = pd.concat(dic_poll_par_date,axis=0)
                                    # # output_file.to_feather(join(root_dir,f'output/patients_polluants_{nb_file_done}.feather'))
                                    # feather.write_feather(output_file[['pseudo_provisoire','Date', 'PM25', 'PM10', 'NO2', 'O3']], join(root_dir,f'output/patients_polluants_{nb_file_done}.feather'))
                                    # i = nb_file_done - 100 
                                    # os.remove(join(root_dir,f'output/patients_polluants_{i}.feather'))
                                    insert_table(dic_poll_par_date)
                                    
                                    
                if 'chronique' not in yeardir : 
                    if year_path not in treated_paths:
                        
                        print(f'Starting directory {year_path}')
                        treated_paths.add(year_path)

                        for file in tqdm(os.listdir(year_path)):

                            if file.endswith('.nc') and file not in treated_polls:  

                                path_to_file = os.path.join(year_path, file)

                                if file.split('_')[0] =='horair':
                                    gdf_processed = process_raster_multibands(path_to_file, gdf)    
                                    dic_poll_par_date[file] = gdf_processed
                                    nb_file_done +=1 
                                else :
                                    gdf_processed = process_raster(path_to_file, gdf)   
                                    dic_poll_par_date[file] = gdf_processed
                                    nb_file_done +=1 
                                treated_polls.add(path_to_file)

                                    
                            if nb_file_done%100 == 0 : 
                                # output_file = pd.concat(dic_poll_par_date,axis=0)[['pseudo_provisoire','Date', 'PM25', 'PM10', 'NO2', 'O3']]
                                # # output_file.to_feather(join(root_dir,f'output/patients_polluants_{nb_file_done}.feather'))
                                # feather.write_feather(output_file, join(root_dir,f'output/patients_polluants_{nb_file_done}.feather'))
                                # i = nb_file_done - 100 
                                # os.remove(join(root_dir,f'output/patients_polluants_{i}.feather'))
                                insert_table(dic_poll_par_date)



Starting directory ../analyse_clinique/data/airparif_dir/airparif/pollution\2017


  7%|▋         | 99/1460 [03:45<51:40,  2.28s/it]  


TypeError: insert_table() missing 1 required positional argument: 'dic_poll_par_date'

In [ ]:
    root_dir = '../analyse_clinique/data/airparif_dir/airparif/'     
    table_name = '202405_patients_pollution'

    dic_poll_par_date = {}
    treated_paths = set()
    treated_polls = set()
    nb_file_done = 0 
    for polldir in os.listdir(root_dir):
        poll_path = join(root_dir,polldir)

        if 'pollution' in polldir:

            for yeardir in os.listdir(poll_path):
                year_path = join(poll_path,yeardir)

                if os.path.isdir(year_path) : 

                    if 'chronique' in yeardir : 

                        for poll in os.listdir(year_path) :               
                            path_to_poll = join(year_path,poll)

                            if path_to_poll not in treated_paths:
                                print(f'Starting directory {path_to_poll}')
                                treated_paths.add(path_to_poll)

                                for file in os.listdir(path_to_poll):

                                    if file.endswith('.nc') and file not in treated_polls:  

                                        path_to_file = os.path.join(path_to_poll, file)

                                        if file.split('_')[0] =='horair':
                                             print(path_to_file,file)
                                        else :
                                            print(path_to_file,file)

                    if 'chronique' not in yeardir : 
                        if year_path not in treated_paths:

                            print(f'Starting directory {year_path}')
                            treated_paths.add(year_path)

                            for file in os.listdir(year_path):

                                if file.endswith('.nc') and file not in treated_polls:  

                                    path_to_file = os.path.join(year_path, file)

                                    if file.split('_')[0] =='horair':
                                        print(path_to_file,file)

                                    else :
                                        print(path_to_file,file)


Starting directory ../../data/test/pollution/2017
../../data/test/pollution/2017/NO2_maxJ_IDF_20170104.nc NO2_maxJ_IDF_20170104.nc
Starting directory ../../data/test/pollution/2018
../../data/test/pollution/2018/PM25_meanJ_IDF_20180317.nc PM25_meanJ_IDF_20180317.nc
Starting directory ../../data/test/pollution/2019
../../data/test/pollution/2019/PM10_meanJ_IDF_20191121.nc PM10_meanJ_IDF_20191121.nc
Starting directory ../../data/test/pollution_chronique/Pollution_chronique_2017/O3
../../data/test/pollution_chronique/Pollution_chronique_2017/O3/horair_IDF_20170105.nc horair_IDF_20170105.nc
Starting directory ../../data/test/pollution_chronique/Pollution_chronique_2017/NO2_PM10_PM25
../../data/test/pollution_chronique/Pollution_chronique_2017/NO2_PM10_PM25/horair_IDF_20170105.nc horair_IDF_20170105.nc
Starting directory ../../data/test/pollution_chronique/Pollution_chronique_2019/O3
../../data/test/pollution_chronique/Pollution_chronique_2019/O3/O3_IDF_20190507.nc O3_IDF_20190507.nc
Starti

In [ ]:
Starting directory ../../data/test/pollution_chronique/Pollution_chronique_2017/O3
../../data/test/pollution_chronique/Pollution_chronique_2017/O3/horair_IDF_20170105.nc horair_IDF_20170105.nc


Starting directory ../../data/test/pollution_chronique/Pollution_chronique_2017/NO2_PM10_PM25
../../data/test/pollution_chronique/Pollution_chronique_2017/NO2_PM10_PM25/horair_IDF_20170105.nc horair_IDF_20170105.nc


Starting directory ../../data/test/pollution_chronique/Pollution_chronique_2019/O3
../../data/test/pollution_chronique/Pollution_chronique_2019/O3/O3_IDF_20190507.nc O3_IDF_20190507.nc


Starting directory ../../data/test/pollution_chronique/Pollution_chronique_2019/NO2_PM10_PM25
../../data/test/pollution_chronique/Pollution_chronique_2019/NO2_PM10_PM25/horair_IDF_20190420.nc horair_IDF_20190420.nc


Starting directory ../../data/test/pollution_chronique/Pollution_chronique_2018/O3
../../data/test/pollution_chronique/Pollution_chronique_2018/O3/O3_IDF_20180209.nc O3_IDF_20180209.nc


Starting directory ../../data/test/pollution_chronique/Pollution_chronique_2018/NO2_PM10_PM25
../../data/test/pollution_chronique/Pollution_chronique_2018/NO2_PM10_PM25/horair_IDF_20180226.nc horair_IDF_20180226.nc

In [ ]:
file_name = 'PM25_meanJ_IDF_20180317.nc'

if ('O3' in file_name) and ('maxJ' not in file_name) :
    print('yes')
else : 
    print('no')

no


In [ ]:
dataframe = pd.DataFrame({"pseudo_provisoire": [1, 2, 3], "date": ['2024-01-01','2024-01-02','2024-01-03'],"PM10": [1, 2, 3]}) #,"NO2": [1, 2, 3],"O3": [1, 2, 3],

cols = dataframe.columns.tolist() 
cols.remove('pseudo_provisoire')
cols.remove('date')
print(cols[0])

PM10


In [ ]:
dict_test = {"pseudo_provisoire": [1, 2, 3], "date": ['2024-01-01','2024-01-02','2024-01-03'],"PM10": [1, 2, 3]}
# dict_test[dict_test.keys[0]]
list(dict_test.keys())[0]

'pseudo_provisoire'

In [ ]:
dict_test[list(dict_test.keys())[1:]]

TypeError: unhashable type: 'list'

In [ ]:
l = list(dict_test.keys())
d3 = { key: dict_test[key] for key in dict_test.keys() if key in l[1:]}

dict_test[l[0]]
d3

{'date': ['2024-01-01', '2024-01-02', '2024-01-03'], 'PM10': [1, 2, 3]}

In [ ]:
l[1:]

['date', 'PM10']